# LeetCode 3869 — Count Fancy Numbers in a Range bằng Python

- Link: https://leetcode.com/problems/count-fancy-numbers-in-a-range/
- Độ khó: **Hard**
- Chủ đề: Digit DP, đếm theo tổng chữ số, liệt kê tổ hợp (số có chữ số đơn điệu ngặt), nguyên lý bao hàm – loại trừ, `f(r) - f(l-1)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countFancy(self, l: int, r: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu định nghĩa: số **tốt** (chữ số tăng ngặt hoặc giảm ngặt) và số **fancy** (tốt, **hoặc** tổng chữ số là số tốt).
2. Viết **Digit DP** đầy đủ trạng thái `(vị trí, tổng chữ số, chữ số trước, trạng thái đơn điệu, chặn trên)`.
3. Nhận ra số tốt chỉ có **1524** số (liệt kê bằng tập con chữ số) ⇒ tách bài thành: đếm theo **tổng chữ số** + xử lý riêng tập nhỏ — dùng **bao hàm–loại trừ** `|A ∪ B| = |B| + |A \ B|`.
4. Đếm số `x ≤ N` có tổng chữ số `s` bằng bảng `ways[k][t]` (số xâu `k` chữ số có tổng `t`).
5. Kiểm thử với duyệt trực tiếp.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Một số nguyên là **tốt** nếu dãy chữ số của nó **tăng ngặt** hoặc **giảm ngặt**. Mọi số một chữ số đều tốt.
- Một số là **fancy** nếu nó tốt **hoặc** tổng chữ số của nó là số tốt.

Đếm số fancy trong `[l, r]`.

**Ví dụ 1:** `l = 8, r = 10` → `3` (8, 9 một chữ số; 10 có chữ số `1 > 0` giảm ngặt).

**Ví dụ 2:** `l = 12340, r = 12341` → `1` (12340 có tổng 10 — tốt; 12341 có tổng 11 — không tốt).

**Ví dụ 3:** `l = r = 123456788` → `0` (tổng 44 — không tốt).

**Ràng buộc:** `1 ≤ l ≤ r ≤ 10^15`.

In [1]:
from functools import cache
from itertools import combinations
from bisect import bisect_right
import random, time

examples = [(8, 10, 3), (12340, 12341, 1), (123456788, 123456788, 0)]
for l, r, exp in examples:
    print(l, r, "-> kỳ vọng", exp)

8 10 -> kỳ vọng 3
12340 12341 -> kỳ vọng 1
123456788 123456788 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Kiểm tra số tốt

Chuyển sang xâu, kiểm tra mọi cặp kề nhau cùng tăng ngặt hoặc cùng giảm ngặt.

In [2]:
def is_good(x):
    s = str(x)
    if len(s) == 1:
        return True
    inc = all(s[i] < s[i + 1] for i in range(len(s) - 1))
    dec = all(s[i] > s[i + 1] for i in range(len(s) - 1))
    return inc or dec

def digit_sum(x):
    return sum(map(int, str(x)))

def is_fancy(x):
    return is_good(x) or is_good(digit_sum(x))

print([x for x in range(1, 40) if not is_fancy(x)])
print(is_good(10), is_good(11), is_good(135), is_good(123), digit_sum(12340))

[]
True False True True 10


Số fancy **rất dày**: mọi số `< 100` đều fancy (số hai chữ số khác nhau là tốt; số có hai chữ số bằng nhau `aa` có tổng `2a` — một chữ số hoặc `1x` với `x ≠ 1`… đều tốt). Số **không** fancy đầu tiên xuất hiện ở hàng trăm. Ô dưới liệt kê vài số không fancy đầu tiên — giúp chọn ca biên cho bộ test.

In [3]:
not_fancy = [x for x in range(1, 1000) if not is_fancy(x)]
print("số không fancy đầu tiên:", not_fancy[:15], "| tổng số trong [1, 999]:", len(not_fancy))
print("tổng chữ số của chúng:", sorted({digit_sum(x) for x in not_fancy}))

số không fancy đầu tiên: [119, 155, 164, 173, 182, 191, 209, 218, 227, 254, 263, 272, 281, 290, 308] | tổng số trong [1, 999]: 64
tổng chữ số của chúng: [11, 22]


### 3.2. Tổng chữ số nhỏ ⇒ tập "tổng tốt" nhỏ

Với `x ≤ 10^15` (16 chữ số), tổng chữ số `≤ 9·15 = 135` (chỉ `10^15` có 16 chữ số nhưng tổng là 1). Tính sẵn tập `GOOD_SUMS = {s ≤ 135 : s tốt}`.

In [4]:
MAX_SUM = 9 * 16
GOOD_SUMS = [s for s in range(1, MAX_SUM + 1) if is_good(s)]
print(len(GOOD_SUMS), GOOD_SUMS)

103 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 123, 124, 125, 126, 127, 128, 129, 134, 135, 136, 137, 138, 139]


### 3.3. Liệt kê mọi số tốt bằng tập con chữ số

- Tăng ngặt (≥ 2 chữ số): chọn tập con `≥ 2` phần tử của `{1..9}` (không có 0 vì 0 không đứng đầu), viết tăng dần.
- Giảm ngặt (≥ 2 chữ số): tập con `≥ 2` phần tử của `{0..9}`, viết giảm dần (chữ số đầu là lớn nhất nên ≠ 0).
- Một chữ số: `1..9`.

Tổng `9 + (2^9 - 1 - 9) + (2^{10} - 1 - 10) = 1524` số.

In [5]:
GOOD_NUMS = list(range(1, 10))
for k in range(2, 10):
    for c in combinations(range(1, 10), k):
        GOOD_NUMS.append(int("".join(map(str, c))))
for k in range(2, 11):
    for c in combinations(range(9, -1, -1), k):
        GOOD_NUMS.append(int("".join(map(str, c))))
GOOD_NUMS.sort()
print(len(GOOD_NUMS), GOOD_NUMS[:15], "... lớn nhất:", GOOD_NUMS[-1])
print("đều tốt:", all(is_good(x) for x in GOOD_NUMS), "| không trùng:", len(set(GOOD_NUMS)) == len(GOOD_NUMS))

1524 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16] ... lớn nhất: 9876543210
đều tốt: True | không trùng: True


### 3.4. Bảng `ways[k][t]`: số xâu `k` chữ số (cho phép 0 đầu) có tổng `t`

`ways[0][0] = 1`; `ways[k][t] = Σ_{d=0..9} ways[k-1][t-d]`.

In [6]:
MAXLEN = 17
ways = [[0] * (MAX_SUM + 10) for _ in range(MAXLEN)]
ways[0][0] = 1
for k in range(1, MAXLEN):
    for t in range(MAX_SUM + 10):
        ways[k][t] = sum(ways[k - 1][t - d] for d in range(10) if t - d >= 0)
print("xâu 2 chữ số tổng 10:", ways[2][10], "| xâu 3 chữ số tổng 27:", ways[3][27])

xâu 2 chữ số tổng 10: 9 | xâu 3 chữ số tổng 27: 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đặt `A` = tập số tốt, `B` = tập số có tổng chữ số tốt. Số fancy = `A ∪ B`.

- `B` phụ thuộc **chỉ vào tổng chữ số** ⇒ đếm được bằng "đếm số `≤ N` có tổng chữ số `s`" cho từng `s ∈ GOOD_SUMS`.
- `A` rất **nhỏ** (1524 số) ⇒ liệt kê được.
- `|A ∪ B| = |B| + |A \ B|`: cộng thêm các số tốt mà tổng chữ số **không** tốt.

Tính tay ví dụ 2: `12340` không tốt nhưng tổng 10 tốt ⇒ thuộc `B`. `12341` không tốt, tổng 11 không tốt ⇒ không fancy.

Ví dụ một số thuộc `A \ B`: `1357` (tăng ngặt) có tổng `16` (tốt vì `1 < 6`)… thuộc cả hai. Còn `9876` tổng `30` (`3 > 0` tốt). Số `48`: tổng `12` tốt. Số `29`: tổng `11` **không** tốt ⇒ `29 ∈ A \ B`.

In [7]:
A_minus_B = [x for x in GOOD_NUMS if not is_good(digit_sum(x))]
print("số lượng A \\ B:", len(A_minus_B), "| vài ví dụ:", A_minus_B[:12])

số lượng A \ B: 138 | vài ví dụ: [29, 38, 47, 56, 65, 74, 83, 92, 128, 137, 146, 236]


## 5. Cách 1 — Brute Force: duyệt từng số

Đếm `is_fancy(x)` với mọi `x ∈ [l, r]`. `O((r - l) · số chữ số)`.

In [8]:
def brute(l, r):
    return sum(1 for x in range(l, r + 1) if is_fancy(x))

for l, r, exp in examples:
    print(brute(l, r), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
0 kỳ vọng 0


**Nút thắt:** `r - l` tới `10^15`. Cần đếm theo cấu trúc chữ số.

## 6. Ý tưởng thuật toán / bước đột phá

**Hướng 1 — Digit DP đầy đủ:** `f(N)` = số fancy trong `[1, N]`, `đáp án = f(r) - f(l-1)`. Trạng thái `(pos, sum, prev, st, tight)`, với `st` ∈ {chưa bắt đầu / mới 1 chữ số, đang tăng, đang giảm, hỏng}. Ở cuối: nếu `st ≠ hỏng` (số tốt) thì đếm, ngược lại đếm khi `sum` tốt. Số trạng thái ≈ `16 · 136 · 10 · 4 · 2` ≈ 170k — chạy được nhưng nặng trong Python.

**Hướng 2 — Đột phá bằng bao hàm–loại trừ:**

- `|B ∩ [1, N]| = Σ_{s ∈ GOOD_SUMS} cnt(N, s)`, với `cnt(N, s)` = số `x ∈ [0, N]` có tổng chữ số `s` (không tính `x = 0` vì `s ≥ 1`).
- `|(A \ B) ∩ [1, N]|` = đếm trong danh sách đã sắp `A_minus_B` bằng `bisect_right`.

`cnt(N, s)` tính bằng cách đi dọc chữ số của `N` và dùng bảng `ways` — `O(16 · 10)` mỗi `s`.

## 7. Các cách tiếp theo

- **Cách 2:** Digit DP đầy đủ trạng thái (giống doocs) — `O(16 · 136 · 10 · 4 · 10)`.
- **Cách 3:** Bao hàm–loại trừ: `B` đếm bằng bảng `ways`, `A \ B` bằng liệt kê + `bisect` (lời giải nộp chính).
- **Cách 4:** Digit DP chỉ theo **tổng chữ số** (memo `(pos, sum, tight)`) cho `|B|`, kết hợp liệt kê `A \ B`.

### 7.1. Cách 2 — Digit DP đầy đủ

Trạng thái `st`: `0` = chưa có chữ số khác 0 hoặc mới có đúng một chữ số (chưa định hướng), `1` = tăng ngặt, `2` = giảm ngặt, `3` = hỏng. Biến `started` phân biệt số 0 đầu.

In [9]:
def full_digit_dp(l, r):
    def f(N):
        if N <= 0:
            return 0
        s = str(N)

        @cache
        def dfs(pos, total, prev, st, started, tight):
            if pos == len(s):
                if not started:
                    return 0                       # số 0 không tính
                return 1 if st != 3 or is_good(total) else 0
            hi = int(s[pos]) if tight else 9
            res = 0
            for d in range(hi + 1):
                if not started:
                    if d == 0:
                        res += dfs(pos + 1, 0, -1, 0, False, tight and d == hi)
                    else:
                        res += dfs(pos + 1, d, d, 0, True, tight and d == hi)
                    continue
                if st == 0:
                    nst = 1 if d > prev else (2 if d < prev else 3)
                elif st == 1:
                    nst = 1 if d > prev else 3
                elif st == 2:
                    nst = 2 if d < prev else 3
                else:
                    nst = 3
                res += dfs(pos + 1, total + d, d, nst, True, tight and d == hi)
            return res

        return dfs(0, 0, -1, 0, False, True)

    return f(r) - f(l - 1)

for l, r, exp in examples:
    print(full_digit_dp(l, r), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp Cách 2:** số trạng thái `≈ 16 · 136 · 11 · 4 · 2 · 2`, mỗi trạng thái 10 nhánh ⇒ vài triệu phép — chạy được trong vài giây Python. Có thể tối ưu: khi `st = 3` thì `prev` không còn quan trọng (gán `prev = 0` để gộp trạng thái).

### 7.2. Cách 3 — Bao hàm–loại trừ (lời giải chính)

**Giải thích từng đoạn:**

1. `cnt(N, s)`: duyệt chữ số `N` từ trái; tại vị trí `i` với chữ số `c`, thử mọi `d < c` cho vị trí này, phần còn lại `k` chữ số tự do có tổng `s - đã_dùng - d` ⇒ cộng `ways[k][...]`. Cuối cùng cộng 1 nếu chính `N` có tổng `s`.
2. `f(N) = Σ_{s ∈ GOOD_SUMS} cnt(N, s) + bisect_right(A_minus_B, N)`.
3. Đáp án `f(r) - f(l - 1)`.

In [10]:
def cnt_digit_sum(N, s):
    """Số x trong [0, N] có tổng chữ số đúng bằng s."""
    if N < 0:
        return 0
    digits = list(map(int, str(N)))
    L = len(digits)
    total, used = 0, 0
    for i, c in enumerate(digits):
        rest = L - i - 1
        for d in range(c):
            need = s - used - d
            if 0 <= need <= 9 * rest:
                total += ways[rest][need]
        used += c
        if used > s:
            return total
    return total + (1 if used == s else 0)

def inclusion_exclusion(l, r):
    def f(N):
        if N <= 0:
            return 0
        return sum(cnt_digit_sum(N, s) for s in GOOD_SUMS) + bisect_right(A_minus_B, N)
    return f(r) - f(l - 1)

for l, r, exp in examples:
    print(inclusion_exclusion(l, r), "kỳ vọng", exp)
print("kiểm tra cnt_digit_sum:", cnt_digit_sum(100, 1), "(1, 10, 100)", cnt_digit_sum(99, 18), "(chỉ 99)")

3 kỳ vọng 3
1 kỳ vọng 1
0 kỳ vọng 0
kiểm tra cnt_digit_sum: 3 (1, 10, 100) 1 (chỉ 99)


**Độ phức tạp Cách 3:** tiền xử lý `ways` `O(17 · 145 · 10)`, liệt kê 1524 số; mỗi `f(N)` là `|GOOD_SUMS| · 16 · 10 ≈ 10^4` phép ⇒ cực nhanh.

### 7.3. Cách 4 — Digit DP theo tổng chữ số + liệt kê

Thay bảng `ways` bằng DP có nhớ `(pos, sum, tight)` đếm số `≤ N` có tổng chữ số tốt (kiểm tra ở lá), rồi cộng `A \ B`.

In [11]:
GOOD_SUM_SET = set(GOOD_SUMS)

def sum_dp(l, r):
    def f(N):
        if N <= 0:
            return 0
        s = str(N)

        @cache
        def dfs(pos, total, tight):
            if pos == len(s):
                return 1 if total in GOOD_SUM_SET else 0
            hi = int(s[pos]) if tight else 9
            return sum(dfs(pos + 1, total + d, tight and d == hi) for d in range(hi + 1))

        return dfs(0, 0, True) + bisect_right(A_minus_B, N)
    return f(r) - f(l - 1)

for l, r, exp in examples:
    print(sum_dp(l, r), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
0 kỳ vọng 0


In [12]:
class SolutionSumDP:
    def countFancy(self, l: int, r: int) -> int:
        return sum_dp(l, r)

print([SolutionSumDP().countFancy(l, r) for l, r, _ in examples])

[3, 1, 0]


**Độ phức tạp Cách 4:** `16 · 136 · 2` trạng thái × 10 nhánh ≈ `4·10^4` mỗi `f`.

Lưu ý: số `0` có tổng `0 ∉ GOOD_SUM_SET` nên không bị đếm.

## 8. TRACE từng bước

Trace `cnt_digit_sum(N = 125, s = 4)`: các số `≤ 125` có tổng chữ số 4: `4, 13, 22, 31, 40, 103, 112, 121`.

| vị trí | chữ số `c` của N | thử `d < c` | phần còn lại | cộng |
|---|---|---|---|---|
| 0 | 1 | d = 0 | 2 chữ số tự do tổng 4 ⇒ `ways[2][4] = 5` | 5 |
| 1 | 2 | d = 0, 1 (đã dùng 1) | 1 chữ số tổng 3, 2 ⇒ 1 + 1 | 2 |
| 2 | 5 | d = 0..4 (đã dùng 3) | 0 chữ số tổng `1 - d` ⇒ chỉ d = 1 | 1 |
| cuối | – | `N = 125` tổng 8 ≠ 4 | – | 0 |

Tổng `8` ✔.

In [13]:
def trace_cnt(N, s):
    digits = list(map(int, str(N))); L = len(digits); total = used = 0
    for i, c in enumerate(digits):
        rest = L - i - 1; add = 0
        for d in range(c):
            need = s - used - d
            if 0 <= need <= 9 * rest:
                add += ways[rest][need]
        total += add
        print(f"vị trí {i}: c={c}, đã dùng={used}, còn {rest} chữ số tự do -> cộng {add}, tổng={total}")
        used += c
    total += used == s
    print("cuối: chính N có tổng", used, "->", total)
    return total

trace_cnt(125, 4)
print("đếm trực tiếp:", [x for x in range(126) if digit_sum(x) == 4])

vị trí 0: c=1, đã dùng=0, còn 2 chữ số tự do -> cộng 5, tổng=5
vị trí 1: c=2, đã dùng=1, còn 1 chữ số tự do -> cộng 2, tổng=7
vị trí 2: c=5, đã dùng=3, còn 0 chữ số tự do -> cộng 1, tổng=8
cuối: chính N có tổng 8 -> 8
đếm trực tiếp: [4, 13, 22, 31, 40, 103, 112, 121]


In [14]:
# Trace f(N) của Cách 3 với N = 30: phần B (theo tổng) và phần A \ B
N = 30
partB = {s: cnt_digit_sum(N, s) for s in GOOD_SUMS if cnt_digit_sum(N, s)}
print("B: số x ≤ 30 có tổng chữ số tốt, theo tổng:", partB, "-> cộng", sum(partB.values()))
print("A \\ B ≤ 30:", [x for x in A_minus_B if x <= N])
print("f(30) =", sum(partB.values()) + bisect_right(A_minus_B, N), "| vét cạn:", brute(1, 30))

B: số x ≤ 30 có tổng chữ số tốt, theo tổng: {1: 2, 2: 3, 3: 4, 4: 3, 5: 3, 6: 3, 7: 3, 8: 3, 9: 3, 10: 2} -> cộng 29
A \ B ≤ 30: [29]
f(30) = 30 | vét cạn: 30


## 9. Code nộp LeetCode

In [15]:
from itertools import combinations
from bisect import bisect_right

class Solution:
    def countFancy(self, l: int, r: int) -> int:
        def good(x):
            s = str(x)
            return all(a < b for a, b in zip(s, s[1:])) or all(a > b for a, b in zip(s, s[1:]))

        MAXS = 9 * 16
        good_sums = [s for s in range(1, MAXS + 1) if good(s)]
        # ways[k][t]: số xâu k chữ số có tổng t
        ways = [[0] * (MAXS + 1) for _ in range(17)]
        ways[0][0] = 1
        for k in range(1, 17):
            prev, cur = ways[k - 1], ways[k]
            for t in range(MAXS + 1):
                cur[t] = sum(prev[t - d] for d in range(10) if t >= d)
        # số tốt có tổng chữ số KHÔNG tốt
        nums = list(range(1, 10))
        for k in range(2, 10):
            nums += [int("".join(map(str, c))) for c in combinations(range(1, 10), k)]
        for k in range(2, 11):
            nums += [int("".join(map(str, c))) for c in combinations(range(9, -1, -1), k)]
        extra = sorted(x for x in nums if not good(sum(map(int, str(x)))))

        def cnt(N, s):
            digits = list(map(int, str(N)))
            L = len(digits)
            total = used = 0
            for i, c in enumerate(digits):
                rest = L - i - 1
                for d in range(c):
                    need = s - used - d
                    if 0 <= need <= 9 * rest:
                        total += ways[rest][need]
                used += c
                if used > s:
                    return total
            return total + (used == s)

        def f(N):
            if N <= 0:
                return 0
            return sum(cnt(N, s) for s in good_sums) + bisect_right(extra, N)

        return f(r) - f(l - 1)

sol = Solution()
for l, r, exp in examples:
    got = sol.countFancy(l, r)
    print(l, r, "->", got, "✔" if got == exp else "✘")

8 10 -> 3 ✔
12340 12341 -> 1 ✔
123456788 123456788 -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **Bao hàm–loại trừ:** fancy = `A ∪ B` là hợp **rời nhau** của `B` và `A \ B`, nên đếm `|B| + |A \ B|` không trùng, không sót.
2. **Đếm `B`:** `B ∩ [1, N] = ⋃_{s ∈ GOOD_SUMS} {x ≤ N : tổng chữ số = s}` là hợp rời (mỗi số một tổng) ⇒ cộng `cnt(N, s)`. Vì `s ≥ 1` nên `x = 0` không bị đếm.
3. **`cnt(N, s)` đúng:** phân hoạch `x < N` theo vị trí đầu tiên `i` mà chữ số của `x` nhỏ hơn của `N` (như digit DP); phần sau tự do nên đếm bằng `ways`. Cộng thêm `x = N` nếu tổng khớp. Số có ít chữ số hơn được coi là có số 0 đầu — tổng chữ số không đổi.
4. **Liệt kê `A`:** số có chữ số tăng ngặt ↔ tập chữ số (không chứa 0 nếu ≥ 2 chữ số); giảm ngặt ↔ tập chữ số bất kỳ ≥ 2 phần tử. Song ánh ⇒ liệt kê đủ và không trùng (một số ≥ 2 chữ số không thể vừa tăng vừa giảm).

## 11. Phân tích Time / Space Complexity

Đặt `D = 16` (số chữ số), `S = 144` (tổng chữ số tối đa), `G = |GOOD_SUMS|`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt | `O((r - l) · D)` | `O(1)` |
| 2. Digit DP đầy đủ | `O(D · S · 10 · 4 · 2 · 10)` | cùng cỡ số trạng thái |
| 3. Bao hàm–loại trừ | `O(D · S · 10 + G · D · 10 + 1524)` | `O(D · S)` |
| 4. Digit DP theo tổng | `O(D · S · 2 · 10)` | `O(D · S)` |

## 12. Bộ test

In [16]:
def run_all(l, r, with_brute=True):
    res = [inclusion_exclusion(l, r), sum_dp(l, r), Solution().countFancy(l, r)]
    if r <= 2 * 10**6:
        res.append(full_digit_dp(l, r))
    if with_brute:
        res.append(brute(l, r))
    return res

for l, r, exp in examples:
    assert all(v == exp for v in run_all(l, r))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [17]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng duyệt)
cases = [(1, 1, 1), (1, 9, 9), (10, 10, 1), (11, 11, 1), (1, 100, 100), (209, 209, 0), (10**15, 10**15, 1)]
for l, r, exp in cases:
    r_all = run_all(l, r, with_brute=(r - l <= 10**5))
    assert len(set(r_all)) == 1, (l, r, r_all)
    if exp is not None:
        assert r_all[0] == exp
    print((l, r), "->", r_all[0])

(1, 1) -> 1
(1, 9) -> 9
(10, 10) -> 1
(11, 11) -> 1
(1, 100) -> 100
(209, 209) -> 0
(1000000000000000, 1000000000000000) -> 1


Ghi chú: `11` không tốt nhưng tổng `2` tốt ⇒ fancy (dễ đoán nhầm là 0). Thực tế **mọi** số từ 1 tới 100 đều fancy. `209`: chữ số `2, 0, 9` không đơn điệu, tổng `11` không tốt ⇒ không fancy. Ca `10^15`: chữ số `1` rồi 15 chữ số `0` — không đơn điệu ngặt, nhưng tổng chữ số `1` là số tốt ⇒ fancy.

In [18]:
# 12.3 phản ví dụ: cộng |A| + |B| (quên trừ phần giao) sẽ đếm trùng
def wrong_double_count(l, r):
    def f(N):
        if N <= 0: return 0
        return sum(cnt_digit_sum(N, s) for s in GOOD_SUMS) + bisect_right(GOOD_NUMS, N)   # SAI
    return f(r) - f(l - 1)

print("sai:", wrong_double_count(1, 100), "| đúng:", Solution().countFancy(1, 100), "| vét cạn:", brute(1, 100))

sai: 182 | đúng: 100 | vét cạn: 100


In [19]:
# 12.4 đối chiếu ngẫu nhiên với duyệt
rng = random.Random(3869)
for _ in range(120):
    l = rng.randint(1, 10**rng.randint(1, 12))
    r = l + rng.randint(0, 3000)
    res = run_all(l, r)
    assert len(set(res)) == 1, (l, r, res)
print("120 đoạn ngắn: OK")
for _ in range(100):
    l = rng.randint(1, 10**15); r = rng.randint(l, 10**15)
    res = run_all(l, r, with_brute=False)
    assert len(set(res)) == 1, (l, r, res)
print("100 đoạn lớn (so 3 cách nhanh): OK")

120 đoạn ngắn: OK


100 đoạn lớn (so 3 cách nhanh): OK


In [20]:
# 12.5 tốc độ
t0 = time.time()
for _ in range(50):
    Solution().countFancy(1, 10**15)
print("50 lần gọi trên [1, 10^15]:", round(time.time() - t0, 3), "s | đáp án:", Solution().countFancy(1, 10**15))

50 lần gọi trên [1, 10^15]: 0.174 s | đáp án: 907441159188136


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm trùng phần giao `A ∩ B`

Đã thấy ở 12.3. **Sửa:** chỉ cộng các số tốt có tổng chữ số **không** tốt.

### Lỗi 2 — Coi 0 là chữ số đứng đầu được trong số tăng ngặt

Nếu liệt kê tập con của `{0..9}` cho số tăng ngặt, ta tạo ra "`012`" = `12` — trùng với số đã có (tập `{1, 2}`), làm đếm trùng.

In [21]:
wrong_inc = [int("".join(map(str, c))) for k in range(2, 11) for c in combinations(range(0, 10), k)]
print("có trùng:", len(wrong_inc) != len(set(wrong_inc)), "| ví dụ 012 ->", int("012"))

có trùng: True | ví dụ 012 -> 12


### Lỗi 3 — Digit DP coi số 0 đầu là chữ số thật

Nếu không có cờ `started`, số `7` được xem như `0…07` — dãy `0, 0, …, 7` **không** tăng ngặt (vì `0 = 0`), nên bị đánh giá sai là không tốt.

In [22]:
def wrong_no_started(N):
    s = str(N).zfill(4)
    inc = all(s[i] < s[i + 1] for i in range(3)); dec = all(s[i] > s[i + 1] for i in range(3))
    return inc or dec

print("7 được coi là tốt? sai:", wrong_no_started(7), "| đúng:", is_good(7))

7 được coi là tốt? sai: False | đúng: True


### Lỗi 4 — Kiểm tra tổng chữ số tốt bằng "không chia hết cho 11"

Mẹo `s % 11 != 0` chỉ đúng cho `s < 100`. Với `s ≥ 100` (ví dụ `s = 100, 120, 132`) phải kiểm tra đầy đủ.

In [23]:
print([(s, s % 11 != 0, is_good(s)) for s in (100, 120, 123, 132, 135)])

[(100, True, False), (120, True, False), (123, True, True), (132, False, False), (135, True, True)]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đếm trong `[l, r]` tới `10^15`** ⇒ `f(r) - f(l - 1)` + digit DP.
- **Điều kiện là "A hoặc B"** mà `A` nhỏ (liệt kê được) ⇒ đếm `B` bằng DP, cộng `A \ B` liệt kê — tránh DP với trạng thái phức tạp.
- **Số có chữ số đơn điệu ngặt** ⇔ **tập con chữ số** ⇒ chỉ có khoảng `2^{10}` số.
- Tổng chữ số của số `≤ 10^15` nhỏ (`≤ 135`) ⇒ trạng thái DP theo tổng rất gọn.

## 15. Biến thể của bài

1. **Fancy nếu tích chữ số tốt**: thay "tổng" bằng "tích" (trạng thái tích có ít giá trị khác nhau — dạng `2^a 3^b 5^c 7^d`).
2. **Không giảm / không tăng** (không ngặt): số lượng lớn hơn nhiều (tổ hợp lặp), cần digit DP thật sự.
3. **Fancy nếu tổng chữ số lặp lại tới một chữ số** (digital root) thuộc tập cho trước: chỉ phụ thuộc `x mod 9`.
4. **Đếm số tốt trong `[l, r]`**: chỉ cần `bisect` trên danh sách 1524 số.

In [24]:
# Biến thể 4: đếm số tốt trong [l, r]
def count_good(l, r):
    return bisect_right(GOOD_NUMS, r) - bisect_right(GOOD_NUMS, l - 1)

print(count_good(1, 100), sum(1 for x in range(1, 101) if is_good(x)))
print("số tốt trong [1, 10^15]:", count_good(1, 10**15))

90 90
số tốt trong [1, 10^15]: 1524


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt | Kiểm tra từng số | `O((r-l)·D)` | `O(1)` | Kiểm thử |
| 2. Digit DP đầy đủ | `(pos, sum, prev, st, tight)` | ~`10^6–10^7` | lớn | Khuôn tổng quát |
| 3. Bao hàm–loại trừ | `B` bằng `ways`, `A\B` liệt kê | ~`10^4` | `O(D·S)` | **Lời giải chuẩn** |
| 4. DP theo tổng + liệt kê | memo `(pos, sum, tight)` | ~`4·10^4` | `O(D·S)` | Khi không muốn tự viết `cnt` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countFancy(10, 20)` = ?

**Bài 2 (tính tay).** Có bao nhiêu số tốt có đúng 3 chữ số?

**Bài 3 (điền TODO).** Viết `cnt_digit_sum_range(l, r, s)` đếm số trong `[l, r]` có tổng chữ số `s`.

**Bài 4 (tự viết lại).** Viết lại `f(N)` của Cách 3 nhưng thay `bisect_right(A_minus_B, N)` bằng vòng lặp đếm (để thấy tập nhỏ đến mức nào).

**Bài 5 (mini challenge).** Biến thể: số **fancy₂** nếu nó tốt **hoặc** **tích** chữ số của nó (khác 0) là số tốt. Đếm trong `[1, N]` với `N ≤ 10^6` bằng digit DP theo tích và kiểm chứng duyệt.

In [25]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def cnt_digit_sum_range(l, r, s):
    # TODO: cnt_digit_sum(r, s) - cnt_digit_sum(l - 1, s)
    pass

In [26]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_fancy2(N):
    # TODO: digit DP trạng thái (pos, product, started, tight); lá: tốt nếu tích tốt; cộng số tốt có tích không tốt
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) 10 tốt (giảm). (2) 11 không, tổng 2 tốt ⇒ fancy; tương tự 12 (tăng)… (3) Kiểm tra từng số: chỉ những số có tổng 11 hoặc 22… không tốt mà không đơn điệu mới loại — trong `[10, 20]` mọi số đều fancy ⇒ `11`.

**Bài 2.** (1) Tăng ngặt 3 chữ số: `C(9, 3) = 84`. (2) Giảm ngặt 3 chữ số: `C(10, 3) = 120`. (3) Tổng `204`.

**Bài 3.** (1) Dùng `cnt_digit_sum`. (2) Hiệu hai tiền tố. (3) `l = 1` thì `l - 1 = 0` có tổng 0.

**Bài 4.** (1) `sum(1 for x in A_minus_B if x <= N)`. (2) `|A \ B|` chỉ vài trăm. (3) So kết quả.

**Bài 5.** (1) Tích chữ số của số `≤ 10^6` thuộc tập nhỏ. (2) Bỏ qua số có chữ số 0 (tích 0 — theo đề bài tự đặt: không tính). (3) Duyệt `1..N` để kiểm chứng.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().countFancy(10, 20) == brute(10, 20)
print("Bài 1:", Solution().countFancy(10, 20))
three = [x for x in GOOD_NUMS if 100 <= x <= 999]
assert len(three) == 204
print("Bài 2:", len(three))

Bài 1: 11
Bài 2: 204


In [28]:
# Bài 3, 4
def cnt_digit_sum_range_sol(l, r, s):
    return cnt_digit_sum(r, s) - cnt_digit_sum(l - 1, s)

assert cnt_digit_sum_range_sol(1, 1000, 1) == 4
assert cnt_digit_sum_range_sol(50, 150, 6) == sum(1 for x in range(50, 151) if digit_sum(x) == 6)

def f_loop(N):
    return sum(cnt_digit_sum(N, s) for s in GOOD_SUMS) + sum(1 for x in A_minus_B if x <= N)

rng = random.Random(4)
for _ in range(50):
    N = rng.randint(1, 10**15)
    assert f_loop(N) - f_loop(0) == Solution().countFancy(1, N)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [29]:
# Bài 5: fancy2 = tốt hoặc tích chữ số (khác 0) là số tốt
def prod_digits(x):
    p = 1
    for ch in str(x): p *= int(ch)
    return p

def is_fancy2(x):
    p = prod_digits(x)
    return is_good(x) or (p > 0 and is_good(p))

def count_fancy2_sol(N):
    s = str(N)
    @cache
    def dfs(pos, prod, started, tight):
        if pos == len(s):
            return 1 if started and prod > 0 and is_good(prod) else 0
        hi = int(s[pos]) if tight else 9; res = 0
        for d in range(hi + 1):
            if not started:
                res += dfs(pos + 1, 1 if d == 0 else d, d > 0, tight and d == hi)
            else:
                res += dfs(pos + 1, prod * d, True, tight and d == hi)
        return res
    by_prod = dfs(0, 1, False, True)
    extra2 = sum(1 for x in GOOD_NUMS if x <= N and not (prod_digits(x) > 0 and is_good(prod_digits(x))))
    return by_prod + extra2

for N in (1, 9, 10, 99, 1000, 54321, 10**6):
    assert count_fancy2_sol(N) == sum(1 for x in range(1, N + 1) if is_fancy2(x)), N
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
GOOD_SUMS = [s for s in 1..144 if good(s)]
GOOD_NUMS = 1..9 + tăng ngặt (tập con {1..9}) + giảm ngặt (tập con {0..9})   # 1524 số
extra = sorted(x in GOOD_NUMS if not good(digit_sum(x)))                      # A \ B
f(N) = Σ_{s in GOOD_SUMS} cnt(N, s) + bisect_right(extra, N)
answer = f(r) - f(l - 1)
```

**Checklist:**

- [ ] Không đếm trùng phần giao (chỉ cộng `A \ B`)?
- [ ] Số tăng ngặt không có chữ số 0; số giảm ngặt được có 0 (ở cuối)?
- [ ] Không đếm `x = 0` (tổng 0 bị loại vì `s ≥ 1`)?
- [ ] Kiểm tra tổng chữ số tốt đầy đủ (kể cả `s ≥ 100`)?
- [ ] `f(r) - f(l - 1)` với `f(0) = 0`?